In [74]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, RANSACRegressor, HuberRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error
import logging
from datetime import datetime

In [75]:
# Set up logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler(f"internet_analysis_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger("internet_analysis")

In [76]:
class DataValidator:
    """Class to validate the dataset before analysis"""

    @staticmethod
    def validate_dataset(df, debug=False):
        """
        Validates the dataset for required columns and data quality

        Args:
            df (DataFrame): Dataset to validate
            debug (bool): Flag to enable debug output

        Returns:
            tuple: (is_valid, issues_found)
        """
        issues = []
        required_columns = ['TRIMESTRE', 'VELOCIDAD BAJADA', 'VELOCIDAD SUBIDA',
                            'No. ACCESOS FIJOS A INTERNET', 'TECNOLOGÍA']

        # Check if DataFrame is empty
        if df.empty:
            issues.append("Dataset is empty")
            return False, issues

        # Check for required columns
        missing_columns = [col for col in required_columns if col not in df.columns]
        if missing_columns:
            issues.append(f"Missing required columns: {missing_columns}")
            return False, issues

        # Check for null values in critical columns
        for col in required_columns:
            null_count = df[col].isnull().sum()
            if null_count > 0:
                issues.append(f"Column {col} has {null_count} null values")

        # Check for data type consistency
        try:
            # Ensure numeric columns are actually numeric
            pd.to_numeric(df['VELOCIDAD BAJADA'])
            pd.to_numeric(df['VELOCIDAD SUBIDA'])
            pd.to_numeric(df['No. ACCESOS FIJOS A INTERNET'])

            # Ensure TRIMESTRE is numeric and sequential
            trimesters = pd.to_numeric(df['TRIMESTRE']).sort_values().unique()
            if len(trimesters) < 2:
                issues.append("Not enough unique time periods for trend analysis")
                # This is a special case - not a critical validation error
                # We'll handle it separately in procesar_municipio

        except Exception as e:
            issues.append(f"Data type consistency error: {str(e)}")

        # Check for negative values in metrics that should be positive
        if (df['VELOCIDAD BAJADA'] < 0).any():
            issues.append("Found negative values in VELOCIDAD BAJADA")
        if (df['VELOCIDAD SUBIDA'] < 0).any():
            issues.append("Found negative values in VELOCIDAD SUBIDA")
        if (df['No. ACCESOS FIJOS A INTERNET'] < 0).any():
            issues.append("Found negative values in No. ACCESOS FIJOS A INTERNET")

        # For trend analysis timing issue, we'll treat this differently
        # We'll consider it valid but flag the issue
        critical_issues = [issue for issue in issues
                           if "Not enough unique time periods" not in issue]

        is_valid = len(critical_issues) == 0

        if debug and issues:
            logger.warning(f"Data validation issues found: {issues}")

        return is_valid, issues

In [77]:
class ModelSelector:
    """Class to select and train prediction models"""

    @staticmethod
    def train_model(X, y, model_type='linear', debug=False):
        """
        Trains a model based on the specified type

        Args:
            X (array): Features (typically time/trimesters)
            y (array): Target values to predict
            model_type (str): Type of model ('linear', 'ransac', 'huber', 'forest')
            debug (bool): Flag to enable debug output

        Returns:
            tuple: (model, metrics) where metrics is a dict of performance metrics
        """
        if len(X) < 3:
            if debug:
                logger.warning(f"Not enough data points ({len(X)}) for reliable model training")
            return None, {"error": "Not enough data points"}

        # Reshape X if it's 1D
        if len(X.shape) == 1:
            X = X.reshape(-1, 1)

        # Dictionary to map model_type to actual model
        models = {
            'linear': LinearRegression(),
            'ransac': RANSACRegressor(random_state=42),
            'huber': HuberRegressor(epsilon=1.35),
            'forest': RandomForestRegressor(n_estimators=50, random_state=42)
        }

        try:
            # Get the model
            model = models.get(model_type, models['linear'])

            # Fit the model
            model.fit(X, y)

            # Make predictions
            y_pred = model.predict(X)

            # Calculate metrics
            r2 = r2_score(y, y_pred)
            mse = mean_squared_error(y, y_pred)
            rmse = np.sqrt(mse)

            # Get slope if it's a linear model
            slope = None
            if hasattr(model, 'coef_'):
                slope = model.coef_[0]

            metrics = {
                'r2': r2,
                'mse': mse,
                'rmse': rmse,
                'slope': slope,
                'model_type': model_type
            }

            if debug:
                logger.info(f"Model trained: {model_type}, R² = {r2:.4f}, RMSE = {rmse:.4f}")
                if slope is not None:
                    logger.info(f"Slope: {slope:.4f}")

            return model, metrics

        except Exception as e:
            if debug:
                logger.error(f"Error training model: {str(e)}")
            return None, {"error": str(e)}

In [78]:
def analizar_tecnologias(df, output_dir='output', debug=False):
    """
    Analyze technology trends by municipality

    Args:
        df (DataFrame): Dataset to analyze
        output_dir (str): Directory to save output files
        debug (bool): Flag to enable debug output

    Returns:
        tuple: (stats_dataframe, plot_path, trends_dict)
    """
    # Ensure output directory exists
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    # Extract location from DataFrame or use a default
    ubicacion = df['UBICACION'].iloc[0] if 'UBICACION' in df.columns else 'Ubicación No Especificada'

    # Validate dataset
    validator = DataValidator()
    is_valid, issues = validator.validate_dataset(df, debug=debug)

    if not is_valid:
        logger.error(f"Invalid dataset for {ubicacion}: {issues}")
        return None, None, None

    # Check if dataset has enough time series points
    num_trimestres = df['TRIMESTRE'].nunique()
    if num_trimestres < 2:
        if debug:
            logger.warning(f"{ubicacion}: Not enough time series points for technology analysis (found {num_trimestres})")
        return None, None, None

    if debug:
        logger.info(f"Analyzing technologies for {ubicacion}")
        logger.info(f"Dataset shape: {df.shape}")
        logger.info(f"Unique technologies: {df['TECNOLOGÍA'].unique()}")
        logger.info(f"Time periods: {df['TRIMESTRE'].unique()}")

    try:
        # Calculate statistics by trimester and technology
        stats_por_trimestre = df.groupby(['TRIMESTRE', 'TECNOLOGÍA']).agg({
            'VELOCIDAD BAJADA': ['mean', 'std'],
            'VELOCIDAD SUBIDA': ['mean', 'std'],
            'No. ACCESOS FIJOS A INTERNET': ['mean', 'std']
        }).reset_index()

        # Rename columns for clarity
        stats_por_trimestre.columns = [
            'TRIMESTRE', 'TECNOLOGÍA',
            'VELOCIDAD_BAJADA_MEAN', 'VELOCIDAD_BAJADA_STD',
            'VELOCIDAD_SUBIDA_MEAN', 'VELOCIDAD_SUBIDA_STD',
            'ACCESOS_MEAN', 'ACCESOS_STD'
        ]

        # Replace NaN with 0 for std when there's only one data point
        stats_por_trimestre = stats_por_trimestre.fillna(0)

        if debug:
            logger.info(f"Generated stats table with shape: {stats_por_trimestre.shape}")

        # Create figure
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 12))

        # Dictionary to store model results
        tendencias_tecnologia = {}

        # Graficar accesos por tecnología
        for tecnologia in df['TECNOLOGÍA'].unique():
            datos_tech = stats_por_trimestre[stats_por_trimestre['TECNOLOGÍA'] == tecnologia]

            # Skip if not enough data points
            if len(datos_tech) <= 1:
                if debug:
                    logger.warning(f"Not enough data points for {tecnologia} in {ubicacion}")
                continue

            # Plot data points
            ax1.errorbar(datos_tech['TRIMESTRE'], datos_tech['ACCESOS_MEAN'],
                         yerr=datos_tech['ACCESOS_STD'], fmt='o-', label=f'{tecnologia}')

            # Train regression model
            X = datos_tech['TRIMESTRE'].values
            y = datos_tech['ACCESOS_MEAN'].values

            # Try different models and pick the best one
            model_types = ['linear', 'ransac', 'huber']
            best_model = None
            best_metrics = {"r2": -float('inf')}

            for model_type in model_types:
                model, metrics = ModelSelector.train_model(X, y, model_type=model_type, debug=debug)
                if model is not None and metrics.get('r2', -float('inf')) > best_metrics['r2']:
                    best_model = model
                    best_metrics = metrics

            if best_model is not None:
                X_reshaped = X.reshape(-1, 1)
                y_pred = best_model.predict(X_reshaped)
                ax1.plot(X, y_pred, '--', alpha=0.5)

                # Store in tendencies dictionary
                tendencias_tecnologia[tecnologia] = {
                    'model_accesos': best_model,
                    **best_metrics,
                    'accesos_media': datos_tech['ACCESOS_MEAN'].mean(),
                    'accesos_std': datos_tech['ACCESOS_STD'].mean()
                }

                if debug:
                    logger.info(f"Technology {tecnologia} - Access model: {best_metrics['model_type']}, R² = {best_metrics['r2']:.4f}")

        ax1.set_title(f'Accesos por Tecnología en {ubicacion}')
        ax1.set_xlabel('Trimestre')
        ax1.set_ylabel('Número de Accesos (Media ± Std)')
        ax1.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
        ax1.grid(True)
        # Use log scale only if data spans multiple orders of magnitude
        if df['No. ACCESOS FIJOS A INTERNET'].max() / max(df['No. ACCESOS FIJOS A INTERNET'].min(), 1) > 100:
            ax1.set_yscale('log')

        # Graficar velocidades por tecnología
        for tecnologia in df['TECNOLOGÍA'].unique():
            datos_tech = stats_por_trimestre[stats_por_trimestre['TECNOLOGÍA'] == tecnologia]

            # Skip if not enough data points
            if len(datos_tech) <= 1:
                continue

            # Plot data points
            ax2.errorbar(datos_tech['TRIMESTRE'], datos_tech['VELOCIDAD_BAJADA_MEAN'],
                         yerr=datos_tech['VELOCIDAD_BAJADA_STD'], fmt='o-',
                         label=f'{tecnologia} - Bajada')

            # Train regression model
            X = datos_tech['TRIMESTRE'].values
            y = datos_tech['VELOCIDAD_BAJADA_MEAN'].values

            # Try different models and pick the best one
            model_types = ['linear', 'ransac', 'huber']
            best_model = None
            best_metrics = {"r2": -float('inf')}

            for model_type in model_types:
                model, metrics = ModelSelector.train_model(X, y, model_type=model_type, debug=debug)
                if model is not None and metrics.get('r2', -float('inf')) > best_metrics['r2']:
                    best_model = model
                    best_metrics = metrics

            if best_model is not None:
                X_reshaped = X.reshape(-1, 1)
                y_pred = best_model.predict(X_reshaped)
                ax2.plot(X, y_pred, '--', alpha=0.5)

                # Update tendencies dictionary
                if tecnologia in tendencias_tecnologia:
                    tendencias_tecnologia[tecnologia].update({
                        'model_velocidad': best_model,
                        'r2_velocidad': best_metrics['r2'],
                        'velocidad_media': datos_tech['VELOCIDAD_BAJADA_MEAN'].mean(),
                        'velocidad_std': datos_tech['VELOCIDAD_BAJADA_STD'].mean()
                    })
                else:
                    tendencias_tecnologia[tecnologia] = {
                        'model_velocidad': best_model,
                        'r2_velocidad': best_metrics['r2'],
                        'velocidad_media': datos_tech['VELOCIDAD_BAJADA_MEAN'].mean(),
                        'velocidad_std': datos_tech['VELOCIDAD_BAJADA_STD'].mean()
                    }

                if debug:
                    logger.info(f"Technology {tecnologia} - Speed model: {best_metrics['model_type']}, R² = {best_metrics['r2']:.4f}")

        ax2.set_title(f'Velocidades por Tecnología en {ubicacion}')
        ax2.set_xlabel('Trimestre')
        ax2.set_ylabel('Velocidad Bajada (Mbps) (Media ± Std)')
        ax2.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
        ax2.grid(True)
        # Use log scale only if data spans multiple orders of magnitude
        if df['VELOCIDAD BAJADA'].max() / max(df['VELOCIDAD BAJADA'].min(), 1) > 100:
            ax2.set_yscale('log')

        plt.tight_layout()

        # Save the figure
        safe_ubicacion = ubicacion.replace(".", "_").replace(" ", "_").replace("/", "_")
        plot_path = os.path.join(output_dir, f"{safe_ubicacion}_tecnologias.png")
        plt.savefig(plot_path, bbox_inches='tight', dpi=300)
        plt.close()

        if debug:
            logger.info(f"Plot saved to {plot_path}")

        return stats_por_trimestre, plot_path, tendencias_tecnologia

    except Exception as e:
        if debug:
            logger.error(f"Error in analizar_tecnologias for {ubicacion}: {str(e)}")
        return None, None, None


In [79]:
def analizar_velocidades(df, output_dir='output', debug=False):
    """
    Analyze velocity trends by municipality

    Args:
        df (DataFrame): Dataset to analyze
        output_dir (str): Directory to save output files
        debug (bool): Flag to enable debug output

    Returns:
        tuple: (stats_dataframe, plot_path, trends_dict)
    """
    # Ensure output directory exists
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    # Extract location from DataFrame or use a default
    ubicacion = df['UBICACION'].iloc[0] if 'UBICACION' in df.columns else 'Ubicación No Especificada'

    # Validate dataset
    validator = DataValidator()
    is_valid, issues = validator.validate_dataset(df, debug=debug)

    if not is_valid:
        logger.error(f"Invalid dataset for {ubicacion}: {issues}")
        return None, None, None

    # Check if dataset has enough time series points
    num_trimestres = df['TRIMESTRE'].nunique()
    if num_trimestres < 2:
        if debug:
            logger.warning(f"{ubicacion}: Not enough time series points for velocity analysis (found {num_trimestres})")
        return None, None, None

    if debug:
        logger.info(f"Analyzing velocities for {ubicacion}")
        logger.info(f"Dataset shape: {df.shape}")
        logger.info(f"Time periods: {df['TRIMESTRE'].unique()}")

    try:
        # Calculate statistics by trimester, focusing on velocities
        stats_por_trimestre = df.groupby(['TRIMESTRE']).agg({
            'VELOCIDAD BAJADA': ['mean', 'std', 'min', 'max'],
            'VELOCIDAD SUBIDA': ['mean', 'std', 'min', 'max'],
            'No. ACCESOS FIJOS A INTERNET': ['sum']
        }).reset_index()

        # Rename columns for clarity
        stats_por_trimestre.columns = [
            'TRIMESTRE',
            'VELOCIDAD_BAJADA_MEAN', 'VELOCIDAD_BAJADA_STD', 'VELOCIDAD_BAJADA_MIN', 'VELOCIDAD_BAJADA_MAX',
            'VELOCIDAD_SUBIDA_MEAN', 'VELOCIDAD_SUBIDA_STD', 'VELOCIDAD_SUBIDA_MIN', 'VELOCIDAD_SUBIDA_MAX',
            'TOTAL_ACCESOS'
        ]

        # Replace NaN with 0 for std when there's only one data point
        stats_por_trimestre = stats_por_trimestre.fillna(0)

        if debug:
            logger.info(f"Generated velocity stats table with shape: {stats_por_trimestre.shape}")

        # Create figure with 2 subplots
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 12))

        # Plot download speed trends
        trimestres = stats_por_trimestre['TRIMESTRE'].values
        velocidad_bajada = stats_por_trimestre['VELOCIDAD_BAJADA_MEAN'].values
        velocidad_bajada_std = stats_por_trimestre['VELOCIDAD_BAJADA_STD'].values

        # Plot data points for download speed
        ax1.errorbar(trimestres, velocidad_bajada, yerr=velocidad_bajada_std,
                     fmt='o-', color='blue', label='Velocidad Bajada')
        ax1.fill_between(trimestres,
                         stats_por_trimestre['VELOCIDAD_BAJADA_MIN'],
                         stats_por_trimestre['VELOCIDAD_BAJADA_MAX'],
                         alpha=0.2, color='blue')

        # Train model for download speed
        if len(trimestres) > 2:
            # Try different models and pick the best one
            model_types = ['linear', 'ransac', 'huber']
            best_model = None
            best_metrics = {"r2": -float('inf')}

            for model_type in model_types:
                model, metrics = ModelSelector.train_model(
                    trimestres, velocidad_bajada, model_type=model_type, debug=debug
                )
                if model is not None and metrics.get('r2', -float('inf')) > best_metrics['r2']:
                    best_model = model
                    best_metrics = metrics

            if best_model is not None:
                # Predict for visualization
                X_pred = np.linspace(min(trimestres), max(trimestres), 100).reshape(-1, 1)
                y_pred = best_model.predict(X_pred)
                ax1.plot(X_pred, y_pred, '--', color='blue', alpha=0.7)

                # Add projection for next two trimesters
                next_trimesters = np.array([max(trimestres) + 1, max(trimestres) + 2]).reshape(-1, 1)
                future_preds = best_model.predict(next_trimesters)
                ax1.plot(next_trimesters.flatten(), future_preds, 'o--', color='red', alpha=0.7,
                         label='Predicción (2 trimestres)')

                # Add model info to plot
                model_info = f"Modelo: {best_metrics['model_type'].capitalize()}, R² = {best_metrics['r2']:.3f}"
                if best_metrics.get('slope') is not None:
                    model_info += f", Pendiente = {best_metrics['slope']:.2f} Mbps/trimestre"
                ax1.text(0.02, 0.95, model_info, transform=ax1.transAxes,
                         bbox=dict(facecolor='white', alpha=0.8))

                if debug:
                    logger.info(f"Download speed model: {best_metrics['model_type']}, R² = {best_metrics['r2']:.4f}")
                    if best_metrics.get('slope') is not None:
                        logger.info(f"Speed slope: {best_metrics['slope']:.4f} Mbps/trimestre")

        ax1.set_title(f'Tendencia de Velocidad de Bajada en {ubicacion}')
        ax1.set_xlabel('Trimestre')
        ax1.set_ylabel('Velocidad Bajada (Mbps)')
        ax1.legend()
        ax1.grid(True)

        # Plot upload speed trends
        velocidad_subida = stats_por_trimestre['VELOCIDAD_SUBIDA_MEAN'].values
        velocidad_subida_std = stats_por_trimestre['VELOCIDAD_SUBIDA_STD'].values

        # Plot data points for upload speed
        ax2.errorbar(trimestres, velocidad_subida, yerr=velocidad_subida_std,
                     fmt='o-', color='green', label='Velocidad Subida')
        ax2.fill_between(trimestres,
                         stats_por_trimestre['VELOCIDAD_SUBIDA_MIN'],
                         stats_por_trimestre['VELOCIDAD_SUBIDA_MAX'],
                         alpha=0.2, color='green')

        # Train model for upload speed
        if len(trimestres) > 2:
            # Try different models and pick the best one
            model_types = ['linear', 'ransac', 'huber']
            best_model = None
            best_metrics = {"r2": -float('inf')}

            for model_type in model_types:
                model, metrics = ModelSelector.train_model(
                    trimestres, velocidad_subida, model_type=model_type, debug=debug
                )
                if model is not None and metrics.get('r2', -float('inf')) > best_metrics['r2']:
                    best_model = model
                    best_metrics = metrics

            if best_model is not None:
                # Predict for visualization
                X_pred = np.linspace(min(trimestres), max(trimestres), 100).reshape(-1, 1)
                y_pred = best_model.predict(X_pred)
                ax2.plot(X_pred, y_pred, '--', color='green', alpha=0.7)

                # Add projection for next two trimesters
                next_trimesters = np.array([max(trimestres) + 1, max(trimestres) + 2]).reshape(-1, 1)
                future_preds = best_model.predict(next_trimesters)
                ax2.plot(next_trimesters.flatten(), future_preds, 'o--', color='red', alpha=0.7,
                         label='Predicción (2 trimestres)')

                # Add model info to plot
                model_info = f"Modelo: {best_metrics['model_type'].capitalize()}, R² = {best_metrics['r2']:.3f}"
                if best_metrics.get('slope') is not None:
                    model_info += f", Pendiente = {best_metrics['slope']:.2f} Mbps/trimestre"
                ax2.text(0.02, 0.95, model_info, transform=ax2.transAxes,
                         bbox=dict(facecolor='white', alpha=0.8))

                if debug:
                    logger.info(f"Upload speed model: {best_metrics['model_type']}, R² = {best_metrics['r2']:.4f}")
                    if best_metrics.get('slope') is not None:
                        logger.info(f"Speed slope: {best_metrics['slope']:.4f} Mbps/trimestre")

        ax2.set_title(f'Tendencia de Velocidad de Subida en {ubicacion}')
        ax2.set_xlabel('Trimestre')
        ax2.set_ylabel('Velocidad Subida (Mbps)')
        ax2.legend()
        ax2.grid(True)

        plt.tight_layout()

        # Save the figure
        safe_ubicacion = ubicacion.replace(".", "_").replace(" ", "_").replace("/", "_")
        plot_path = os.path.join(output_dir, f"{safe_ubicacion}_velocidades.png")
        plt.savefig(plot_path, bbox_inches='tight', dpi=300)
        plt.close()

        if debug:
            logger.info(f"Velocity plot saved to {plot_path}")

        # Create dictionary with trend analysis results
        tendencias_velocidad = {
            'bajada': {
                'media': np.mean(velocidad_bajada),
                'tendencia': best_metrics.get('slope', 0) if 'best_metrics' in locals() else 0,
                'r2': best_metrics.get('r2', 0) if 'best_metrics' in locals() else 0,
                'modelo': best_metrics.get('model_type', 'N/A') if 'best_metrics' in locals() else 'N/A',
                'prediccion_2_trimestres': future_preds.tolist() if 'future_preds' in locals() else []
            },
            'subida': {
                'media': np.mean(velocidad_subida),
                'tendencia': best_metrics.get('slope', 0) if 'best_metrics' in locals() else 0,
                'r2': best_metrics.get('r2', 0) if 'best_metrics' in locals() else 0,
                'modelo': best_metrics.get('model_type', 'N/A') if 'best_metrics' in locals() else 'N/A',
                'prediccion_2_trimestres': future_preds.tolist() if 'future_preds' in locals() else []
            }
        }

        return stats_por_trimestre, plot_path, tendencias_velocidad

    except Exception as e:
        if debug:
            logger.error(f"Error in analizar_velocidades for {ubicacion}: {str(e)}")
        return None, None, None

In [80]:

def procesar_municipio(ruta_archivo, output_dir='output', debug=False):
    """
    Process a single municipality file with both analysis methods

    Args:
        ruta_archivo (str): Path to the municipality CSV file
        output_dir (str): Directory to save output files
        debug (bool): Flag to enable debug output

    Returns:
        dict: Results of both analyses
    """
    try:
        # Create output directory if it doesn't exist
        if not os.path.exists(output_dir):
            os.makedirs(output_dir)

        # Load the dataset
        df = pd.read_csv(ruta_archivo)

        # Extract municipality name from filename
        nombre_municipio = os.path.basename(ruta_archivo).replace('.csv', '')

        if debug:
            logger.info(f"Processing municipality: {nombre_municipio}")
            logger.info(f"Data shape: {df.shape}")

        # Add UBICACION column if not present
        if 'UBICACION' not in df.columns:
            df['UBICACION'] = nombre_municipio

        # Check if dataset has enough unique time series points
        num_trimestres = df['TRIMESTRE'].nunique()

        # Create basic result structure that will be returned regardless of outcome
        results = {
            'municipio': nombre_municipio,
            'status': 'success',  # Will be updated if needed
            'tecnologias': {
                'stats': None,
                'plot_path': None,
                'trends': None
            },
            'velocidades': {
                'stats': None,
                'plot_path': None,
                'trends': None
            }
        }

        if num_trimestres < 2:
            warning_msg = "Not enough data: dataset contains only one unique time series point"
            logger.warning(f"{nombre_municipio}: {warning_msg}")

            # Create basic stats even without time series
            results.update({
                'status': 'warning',
                'message': warning_msg,
                'tecnologias': {
                    'stats': df.groupby('TECNOLOGÍA').agg({
                        'VELOCIDAD BAJADA': 'mean',
                        'VELOCIDAD SUBIDA': 'mean',
                        'No. ACCESOS FIJOS A INTERNET': 'sum'
                    }).reset_index().to_dict('records') if 'TECNOLOGÍA' in df.columns else None,
                    'plot_path': None,
                    'trends': None
                },
                'velocidades': {
                    'stats': {col: df[col].agg(['mean', 'min', 'max']).to_dict()
                              for col in ['VELOCIDAD BAJADA', 'VELOCIDAD SUBIDA']}
                    if not df.empty else None,
                    'plot_path': None,
                    'trends': None
                }
            })
        else:
            # Run both analyses
            try:
                tech_stats, tech_plot, tech_trends = analizar_tecnologias(df, output_dir=output_dir, debug=debug)
                if tech_stats is not None:
                    results['tecnologias']['stats'] = tech_stats.to_dict('records')
                    results['tecnologias']['plot_path'] = tech_plot
                    results['tecnologias']['trends'] = tech_trends
            except Exception as e:
                if debug:
                    logger.error(f"Error in technology analysis for {nombre_municipio}: {str(e)}")
                results['status'] = 'partial'
                results['tech_error'] = str(e)

            try:
                vel_stats, vel_plot, vel_trends = analizar_velocidades(df, output_dir=output_dir, debug=debug)
                if vel_stats is not None:
                    results['velocidades']['stats'] = vel_stats.to_dict('records')
                    results['velocidades']['plot_path'] = vel_plot
                    results['velocidades']['trends'] = vel_trends
            except Exception as e:
                if debug:
                    logger.error(f"Error in velocity analysis for {nombre_municipio}: {str(e)}")
                results['status'] = 'partial'
                results['vel_error'] = str(e)

        # Save results to JSON
        import json
        results_file = os.path.join(output_dir, f"{nombre_municipio.replace('.', '_').replace(' ', '_').replace('-', '_')}_results.json")

        # We need to remove model objects which are not JSON serializable
        serializable_results = results.copy()
        if serializable_results['tecnologias']['trends']:
            for tech, trend_data in serializable_results['tecnologias']['trends'].items():
                if 'model_accesos' in trend_data:
                    del trend_data['model_accesos']
                if 'model_velocidad' in trend_data:
                    del trend_data['model_velocidad']

        with open(results_file, 'w', encoding='utf-8') as f:
            json.dump(serializable_results, f, ensure_ascii=False, indent=2)

        if debug:
            logger.info(f"Results saved to {results_file}")

        return results

    except Exception as e:
        if debug:
            logger.error(f"Critical error processing {ruta_archivo}: {str(e)}")

        # Try to create a minimal result file even in case of error
        try:
            nombre_municipio = os.path.basename(ruta_archivo).replace('.csv', '')
            error_results = {
                'municipio': nombre_municipio,
                'status': 'error',
                'error': str(e)
            }

            import json
            error_file = os.path.join(output_dir, f"{nombre_municipio.replace('.', '_').replace(' ', '_').replace('-', '_')}_results.json")
            with open(error_file, 'w', encoding='utf-8') as f:
                json.dump(error_results, f, ensure_ascii=False, indent=2)

            if debug:
                logger.info(f"Error results saved to {error_file}")
        except Exception as inner_e:
            if debug:
                logger.error(f"Failed to save error results: {str(inner_e)}")

        return {
            'municipio': os.path.basename(ruta_archivo).replace('.csv', ''),
            'status': 'error',
            'error': str(e)
        }

        # Save results to JSON
        import json
        results_file = os.path.join(output_dir, f"{nombre_municipio.replace('.', '_')}_results.json")

        # We need to remove model objects which are not JSON serializable
        serializable_results = results.copy()
        if serializable_results['tecnologias']['trends']:
            for tech, trend_data in serializable_results['tecnologias']['trends'].items():
                if 'model_accesos' in trend_data:
                    del trend_data['model_accesos']
                if 'model_velocidad' in trend_data:
                    del trend_data['model_velocidad']

        with open(results_file, 'w', encoding='utf-8') as f:
            json.dump(serializable_results, f, ensure_ascii=False, indent=2)

        if debug:
            logger.info(f"Results saved to {results_file}")

        return results

    except Exception as e:
        if debug:
            logger.error(f"Error processing {ruta_archivo}: {str(e)}")

        return {
            'municipio': os.path.basename(ruta_archivo).replace('.csv', ''),
            'status': 'error',
            'error': str(e)
        }

In [83]:
# Example usage
if __name__ == "__main__":
    # Example with a single file
    ruta_archivo = "../../Limpieza/data/subdatasets-ubicacion/AMAZONAS.MIRITI - PARANÁ.csv"
    # Enable debug mode to see detailed logs
    results = procesar_municipio(ruta_archivo, output_dir='resultados', debug=True)

    # Print summary of results
    if results['status'] == 'success':
        print(f"Successfully processed {results['municipio']}")

        # Technology trends
        print("\nTechnology Trends:")
        for tech, trend in results['tecnologias']['trends'].items():
            print(f"  - {tech}:")
            if 'r2_velocidad' in trend:
                print(f"    Velocidad: R²={trend['r2_velocidad']:.4f}, Media={trend['velocidad_media']:.2f} Mbps")
            if 'r2' in trend:
                print(f"    Accesos: R²={trend['r2']:.4f}, Media={trend['accesos_media']:.2f}")

        # Velocity trends
        print("\nVelocity Trends:")
        vel_trends = results['velocidades']['trends']
        print(f"  - Bajada: Media={vel_trends['bajada']['media']:.2f} Mbps, Tendencia={vel_trends['bajada']['tendencia']:.4f}, R²={vel_trends['bajada']['r2']:.4f}")
        print(f"  - Subida: Media={vel_trends['subida']['media']:.2f} Mbps, Tendencia={vel_trends['subida']['tendencia']:.4f}, R²={vel_trends['subida']['r2']:.4f}")

        print(f"\nPlots saved to:")
        print(f"  - {results['tecnologias']['plot_path']}")
        print(f"  - {results['velocidades']['plot_path']}")
    elif results['status'] == 'warning':
        print(f"Warning for {results['municipio']}: {results['message']}")
        print("Basic statistics were generated but no trend analysis or plots were created")
    else:
        print(f"Error processing {results['municipio']}: {results.get('error', 'Unknown error')}")

2025-03-02 19:00:45,953 - internet_analysis - INFO - Processing municipality: AMAZONAS.MIRITI - PARANÁ
2025-03-02 19:00:45,956 - internet_analysis - INFO - Data shape: (2, 5)
2025-03-02 19:00:45,960 - internet_analysis - WARNING - AMAZONAS.MIRITI - PARANÁ: Not enough data: dataset contains only one unique time series point
2025-03-02 19:00:45,985 - internet_analysis - INFO - Results saved to resultados/AMAZONAS_MIRITI___PARANÁ_results.json


Warning for AMAZONAS.MIRITI - PARANÁ: Not enough data: dataset contains only one unique time series point
Basic statistics were generated but no trend analysis or plots were created
